### Requirements

In [ ]:
!pip install pyautogui

In [77]:
# Base Modules
import math
import time
import numpy as np
from collections import deque

# OpenCV
import cv2

# MediaPipe
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# PyAutoGUI
import pyautogui
import ctypes

## Parameters

In [78]:
# Camera Settings
camera_index = 700
resolution = [640, 480]
fps = 60

# Model Settings
model_path = '../models/hand_landmarker.task'
no_of_hands = 1
hand_confidence = 0.7
presence_confidence = 0.6
tracking_confidence = 0.6

# Gesture Settings
pinch_dist_max = 35
click_dist_max = 40
right_click_dist_max = 45
middle_click_dist_max = 35
scroll_pinch_dist_max = 35
scroll_speed = 40           
scroll_deadzone = 4
jitter_deadzone = 8
drag_move_threshold = 15
double_click_time = 0.65

# Screen & Cursor Settings
frame_margin = 90 
smoothing = 5

### Internal Parameters

In [79]:
# Get primary monitor resolution
screen_w, screen_h = pyautogui.size()
pyautogui.FAILSAFE = False  # Avoids crash if cursor bumps screen edges

# Tracking memory variables
prev_x, prev_y = screen_w // 2, screen_h // 2
is_clicked = False  # Debounce lock

# Tracking state variables
last_tap_time = 0
tap_count = 0
prev_scroll_y = 0
is_touching = False 
is_dragging = False
is_right_clicked = False
is_middle_clicked = False

# Optimizations
pyautogui.PAUSE = 0.0
grace_counter = 0
MAX_GRACE_FRAMES = 2
last_valid_landmarks = None
touch_start_pos = (0, 0)
coord_buffer_x = deque(maxlen=5)
coord_buffer_y = deque(maxlen=5)

### MediaPipe

In [80]:
options = vision.HandLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path=model_path),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=no_of_hands,
    min_hand_detection_confidence=hand_confidence,
    min_hand_presence_confidence=presence_confidence,
    min_tracking_confidence=tracking_confidence
)
detector = vision.HandLandmarker.create_from_options(options)

## Air Mouse Logic

In [81]:
def process_air_mouse(frame, hand, w, h):
    global prev_x, prev_y, is_touching, last_tap_time, tap_count, is_dragging
    global is_right_clicked, is_middle_clicked, prev_scroll_y, touch_start_pos
    
    current_time = time.time()
    status_text = "Idle"
    
    # Active bounding box
    cv2.rectangle(frame, (frame_margin, frame_margin), 
                  (w - frame_margin, h - frame_margin), (200, 200, 200), 1)

    if hand is not None:
        # Landmark Indices:
        # 4 = Thumb | 8 = Index | 12 = Middle | 16 = Ring | 20 = Pinky
        thumb_x, thumb_y = int(hand[4].x * w), int(hand[4].y * h)
        idx_x, idx_y = int(hand[8].x * w), int(hand[8].y * h)
        mid_x, mid_y = int(hand[12].x * w), int(hand[12].y * h)
        ring_x, ring_y = int(hand[16].x * w), int(hand[16].y * h)
        pinky_x, pinky_y = int(hand[20].x * w), int(hand[20].y * h)

        # Distance Calculations
        move_pinch_dist = math.hypot(thumb_x - idx_x, thumb_y - idx_y)        # Thumb + Index (Move)
        scroll_pinch_dist = math.hypot(thumb_x - ring_x, thumb_y - ring_y)    # Thumb + Ring (Scroll)
        middle_click_dist = math.hypot(thumb_x - pinky_x, thumb_y - pinky_y)  # Thumb + Pinky (Middle Click)
        left_click_dist = math.hypot(idx_x - mid_x, idx_y - mid_y)           # Index + Middle (Left Click)
        right_click_dist = math.hypot(idx_x - ring_x, idx_y - ring_y)        # Index + Ring (Right Click)

        # Draw hand skeleton dots
        # for lm in hand:
        #     cv2.circle(frame, (int(lm.x * w), int(lm.y * h)), 3, (0, 0, 255), cv2.FILLED)

        # ==================== 1. MIDDLE CLICK (Thumb + Pinky) ==================== #
        if middle_click_dist < middle_click_dist_max:
            if not is_middle_clicked:
                pyautogui.middleClick()
                is_middle_clicked = True
            status_text = "MIDDLE CLICK!"
            
            # Visual feedback (Blue line & circle)
            # cv2.line(frame, (thumb_x, thumb_y), (pinky_x, pinky_y), (255, 0, 0), 2)
            # cv2.circle(frame, (pinky_x, pinky_y), 9, (255, 0, 0), cv2.FILLED)
            # cv2.circle(frame, (thumb_x, thumb_y), 7, (255, 0, 0), cv2.FILLED)

        # ==================== 2. SCROLL MODE (Thumb + Ring) ==================== #
        elif scroll_pinch_dist < scroll_pinch_dist_max:
            is_middle_clicked = False
            status_text = "SCROLL MODE"
            
            # cv2.line(frame, (thumb_x, thumb_y), (ring_x, ring_y), (255, 0, 255), 3)
            # cv2.circle(frame, (ring_x, ring_y), 8, (255, 0, 255), cv2.FILLED)
            # cv2.circle(frame, (thumb_x, thumb_y), 8, (255, 0, 255), cv2.FILLED)

            if prev_scroll_y == 0:
                prev_scroll_y = thumb_y

            dy = prev_scroll_y - thumb_y
            if abs(dy) > scroll_deadzone:
                scroll_units = int(dy * (scroll_speed / 10))
                pyautogui.scroll(scroll_units)
                prev_scroll_y = thumb_y

            if is_dragging:
                pyautogui.mouseUp(button='left')
                is_dragging = False

        # ==================== 3. CURSOR & CLICKS (Thumb + Index) ==================== #
        elif move_pinch_dist < pinch_dist_max:
            is_middle_clicked = False
            prev_scroll_y = 0
            status_text = "Tracking Cursor"

            # 1. Project target position onto monitor
            raw_target_x = np.interp(idx_x, (frame_margin, w - frame_margin), (0, screen_w))
            raw_target_y = np.interp(idx_y, (frame_margin, h - frame_margin), (0, screen_h))

            # 2. Feed into Moving Average Buffer to kill sensor noise
            coord_buffer_x.append(raw_target_x)
            coord_buffer_y.append(raw_target_y)
            
            target_x = sum(coord_buffer_x) / len(coord_buffer_x)
            target_y = sum(coord_buffer_y) / len(coord_buffer_y)

            travel_dist = math.hypot(target_x - prev_x, target_y - prev_y)
            
            # Check contacts
            is_left_touching = left_click_dist < click_dist_max
            is_right_touching = right_click_dist < right_click_dist_max
            
            # Freeze only during tap/right-click if not in drag mode
            should_freeze = (is_right_touching or (is_left_touching and not is_dragging)) and (travel_dist < 8)
            
            # 3. Position Calculation (Deadzone + Steady EMA)
            if should_freeze:
                curr_x, curr_y = prev_x, prev_y
            elif travel_dist < 3.0:  # Deadzone for resting hand
                curr_x, curr_y = prev_x, prev_y
            else:
                # Balanced smoothing: 0.35 steady, 0.60 fast
                smooth_weight = 0.35 if travel_dist < 20 else 0.60
                curr_x = prev_x + smooth_weight * (target_x - prev_x)
                curr_y = prev_y + smooth_weight * (target_y - prev_y)
            
            # Direct OS move
            ctypes.windll.user32.SetCursorPos(int(curr_x), int(curr_y))
            prev_x, prev_y = curr_x, curr_y

            # Visuals for cursor tracking
            # cv2.line(frame, (thumb_x, thumb_y), (idx_x, idx_y), (0, 255, 0), 2)
            # cv2.circle(frame, (idx_x, idx_y), 7, (0, 255, 0), cv2.FILLED)
            # cv2.circle(frame, (thumb_x, thumb_y), 7, (0, 255, 0), cv2.FILLED)

            # --- Right Click (Index + Ring) ---
            if is_right_touching:
                if not is_right_clicked:
                    pyautogui.rightClick()
                    is_right_clicked = True
                status_text = "RIGHT CLICK!"
                # cv2.line(frame, (idx_x, idx_y), (ring_x, ring_y), (0, 165, 255), 2)
                # cv2.circle(frame, (ring_x, ring_y), 9, (0, 165, 255), cv2.FILLED)
            else:
                is_right_clicked = False

            # --- Left Click, Double Click & Movement-Initiated Drag (Index + Middle) ---
            if is_left_touching:
                # cv2.line(frame, (idx_x, idx_y), (mid_x, mid_y), (255, 255, 0), 2)
                # cv2.circle(frame, (mid_x, mid_y), 9, (255, 255, 0), cv2.FILLED)

                if not is_touching:
                    # First frame of contact
                    is_touching = True
                    touch_start_pos = (curr_x, curr_y)
                    
                    if (current_time - last_tap_time) <= double_click_time:
                        pyautogui.doubleClick()
                        tap_count = 0
                        last_tap_time = 0
                        status_text = "DOUBLE CLICK!"
                    else:
                        tap_count = 1
                        last_tap_time = current_time
                else:
                    # Contact maintained: Check if hand moved far enough to instantly drag
                    dist_from_start = math.hypot(curr_x - touch_start_pos[0], curr_y - touch_start_pos[1])
                    if not is_dragging and dist_from_start > drag_move_threshold:
                        pyautogui.mouseDown(button='left')
                        is_dragging = True
                        tap_count = 0  # Cancel single-click pending action
                    
                    if is_dragging:
                        status_text = "DRAGGING / HOLDING!"
            else:
                # Contact released
                if is_dragging:
                    pyautogui.mouseUp(button='left')
                    is_dragging = False
                    tap_count = 0
                is_touching = False

        # ==================== 4. IDLE ==================== #
        else:
            coord_buffer_x.clear()
            coord_buffer_y.clear()
            prev_scroll_y = 0
            is_middle_clicked = False
            is_right_clicked = False
            is_touching = False
            if is_dragging:
                pyautogui.mouseUp(button='left')
                is_dragging = False
                
            # cv2.circle(frame, (idx_x, idx_y), 7, (0, 0, 255), cv2.FILLED)
            # cv2.circle(frame, (thumb_x, thumb_y), 7, (255, 0, 255), cv2.FILLED)

    # Resolve pending single click
    if tap_count == 1 and not is_dragging and (current_time - last_tap_time > double_click_time):
        pyautogui.click()
        tap_count = 0
        status_text = "SINGLE CLICK!"

    return frame, status_text

### Camera Configuration

In [82]:
cap = cv2.VideoCapture(camera_index)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, resolution[0])
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, resolution[1])
cap.set(cv2.CAP_PROP_FPS, fps)

True

# Main Logic

In [83]:
print("Air Mouse Active! Move inside the white box. Press 'q' to stop.")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape

    # MediaPipe inference
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    timestamp_ms = int(time.time() * 1000)
    result = detector.detect_for_video(mp_image, timestamp_ms)

    # --- GRACE BUFFER LOGIC --- #
    hand = None
    if result.hand_landmarks:
        hand = result.hand_landmarks[0]
        last_valid_landmarks = hand
        grace_counter = 0
    elif grace_counter < MAX_GRACE_FRAMES and last_valid_landmarks is not None:
        # Re-use last known landmarks across 1-2 blurred frames
        hand = last_valid_landmarks
        grace_counter += 1

    # Pass the resolved hand (or None) into your gesture processor
    frame, status_text = process_air_mouse(frame, hand, w, h)

    # HUD Status
    #is_active = "Tracking" in status_text or "CLICK" in status_text
    #cv2.putText(frame, status_text, (20, 40), cv2.FONT_HERSHEY_SCRIPT_SIMPLEX, 0.75, 
    #            (0, 255, 0) if is_active else (0, 0, 255), 2)

    cv2.imshow("Air Mouse", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Air Mouse Active! Move inside the white box. Press 'q' to stop.
